# <font color="#418FDE" size="6.5" uppercase>**Klassische Bildmodelle**</font>

>Last update: 20260825.
    
By the end of this Lecture, you will be able to:
- Bereiten kleine Bilddaten als Merkmalsvektoren oder klassische Bildmerkmale vor. 
- Trainieren klassische Klassifikatoren auf Bildmerkmalen ohne Datenleckage. 
- Bewerten Bildmodelle mit Konfusionsmatrix und visualisierten Fehlklassifikationen. 


## **1. Bildmerkmale vorbereiten**

### **1.1. Digits Datensatz laden**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_A/image_01_01.jpg?v=1787650756" width="250">



>* Kleine Graustufenbilder handgeschriebener Ziffern
>* Modelle nutzen Pixelhelligkeiten statt Formen

>* Bilder als Pixelraster mit Helligkeitswerten
>* Zielklassen trennen Eingaben von Bewertungen

>* Beispiele früh auf Plausibilität prüfen
>* Handschriftvariationen beeinflussen die Merkmalsvorbereitung



In [ ]:
#@title Python-Code - Digits Datensatz laden

# Wir laden kleine Ziffernbilder als Lernbeispiel.
# Bildraster und Zielklassen werden getrennt betrachtet.
# Ein Beispielbild zeigt die numerische Bildstruktur.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

# Der Datensatz ist lokal in scikit-learn enthalten.
digits = load_digits()
images = digits.images
targets = digits.target

# Diese Prüfung schützt vor falsch verstandenen Bildformen.
if images.ndim != 3 or images.shape[1:] != (8, 8):
    raise ValueError("Erwartet werden viele Bilder mit jeweils 8 mal 8 Pixeln.")

# Für klassische Modelle wird jedes Bild später flach gemacht.
feature_vectors = images.reshape(images.shape[0], -1)
first_image = images[0]
first_vector = feature_vectors[0]

# Kurze Ausgaben zeigen die wichtigsten Datenbestandteile.
print(f"Anzahl Bilder: {images.shape[0]}")
print(f"Bildform: {images.shape[1]} x {images.shape[2]} Pixel")
print(f"Merkmale pro flachem Bildvektor: {feature_vectors.shape[1]}")
print(f"Zielklasse des gezeigten Bildes: {targets[0]}")
print(f"Erste 10 Pixelwerte im Vektor: {first_vector[:10].astype(int).tolist()}")

# Das Bild macht die Pixelwerte visuell nachvollziehbar.
fig, ax = plt.subplots(figsize=(4, 4))
image_plot = ax.imshow(first_image, cmap="gray_r", vmin=0, vmax=16)

# Achsenbeschriftungen beziehen sich auf Pixelpositionen.
ax.set_title("Erstes Bild aus dem Digits-Datensatz")
ax.set_xlabel("Pixelspalte")
ax.set_ylabel("Pixelzeile")

# Eine Farbskala erklärt die Helligkeitswerte.
fig.colorbar(image_plot, ax=ax, label="Helligkeitswert")
plt.show()



### **1.2. Pixel als Merkmale**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_A/image_01_02.jpg?v=1787650759" width="250">



>* Bilder werden zu geordneten Zahlenfolgen.
>* Geeignet für gleich große, ausgerichtete Bilder.

>* Pixelmerkmale sind einfach und direkt interpretierbar
>* Vorverarbeitung reduziert Empfindlichkeit gegenüber Bildvariationen

>* Pixelwerte skalieren und Dimensionen beachten
>* Nützliche Baseline, aber begrenzt robust



In [ ]:
#@title Python-Code - Pixel als Merkmale

# Dieses Beispiel macht Pixelmerkmale sichtbar.
# Ein Ziffernbild wird zum Merkmalsvektor.
# Die Ausgabe zeigt Form und Wertebereich.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

# Wir nutzen kleine, eingebaute Graustufenbilder von Ziffern.
digits = load_digits()
images = digits.images
labels = digits.target

# Eine einfache Prüfung verhindert missverständliche Formen.
if images.ndim != 3 or images.shape[1:] != (8, 8):
    raise ValueError("Erwartet werden 8x8-Graustufenbilder.")

# Ein einzelnes Bild wird ausgewählt und unverändert betrachtet.
image_index = 0
image = images[image_index]
label = labels[image_index]

# Flatten macht aus dem Raster eine geordnete Zahlenfolge.
pixel_vector = image.reshape(-1)
scaled_vector = pixel_vector / 16.0

print(f"Klasse des Beispielbildes: {label}")
print(f"Bildform als Raster: {image.shape[0]} x {image.shape[1]} Pixel")
print(f"Länge des Pixelvektors: {pixel_vector.size} Merkmale")
print(f"Erste 12 rohe Pixelwerte: {pixel_vector[:12].astype(int).tolist()}")
print(f"Erste 12 skalierte Werte: {np.round(scaled_vector[:12], 2).tolist()}")

# Die Grafik zeigt, wo die Vektorwerte im Bild liegen.
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(image, cmap="gray_r", vmin=0, vmax=16)
ax.set_title("8x8-Ziffer: Pixel werden Merkmale")
ax.set_xlabel("Pixelspalte")
ax.set_ylabel("Pixelzeile")

# Kleine Zahlen markieren die Reihenfolge im Merkmalsvektor.
for row in range(image.shape[0]):
    for col in range(image.shape[1]):
        feature_index = row * image.shape[1] + col
        ax.text(col, row, str(feature_index), ha="center", va="center", fontsize=7)

plt.show()



### **1.3. Histogramme als Merkmale**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_A/image_01_03.jpg?v=1787650757" width="250">



>* Histogramme fassen Intensitätsverteilungen kompakt zusammen
>* Robuster und kleiner als reine Pixelmerkmale

>* Bin-Größe bestimmt Detailgrad und Robustheit
>* Histogramme liefern kompakte Signaturen für Klassifikatoren

>* Regionale Histogramme bewahren räumliche Hinweise
>* Einheitliche Vorverarbeitung macht Merkmale vergleichbar



In [ ]:
#@title Python-Code - Histogramme als Merkmale

# Dieses Beispiel zeigt Histogramme als Bildmerkmale.
# Grauwerte werden zu kompakten Häufigkeiten zusammengefasst.
# Der Plot vergleicht zwei synthetische Ziffernbilder.

import numpy as np
import matplotlib.pyplot as plt

# Zwei kleine synthetische Grauwertbilder werden vorbereitet.
image_size = 16
thin_digit = np.full((image_size, image_size), 230, dtype=np.uint8)
thick_digit = np.full((image_size, image_size), 230, dtype=np.uint8)

# Dunkle Striche bilden einfache ziffernähnliche Muster.
thin_digit[3:13, 7:9] = 35
thin_digit[11:13, 5:11] = 35
thick_digit[3:13, 6:10] = 35

# Ein mittlerer Grauwert simuliert weiche Kanten.
thin_digit[3:13, 6] = 120
thin_digit[3:13, 9] = 120
thick_digit[3:13, 5] = 120
thick_digit[3:13, 10] = 120

# Beide Bilder müssen dieselbe Form besitzen.
if thin_digit.shape != thick_digit.shape:
    raise ValueError("Die Bilder müssen gleich groß sein.")

# Gleiche Histogramm-Grenzen erzeugen vergleichbare Merkmalsvektoren.
bin_edges = np.array([0, 64, 128, 192, 256])
thin_hist, _ = np.histogram(thin_digit, bins=bin_edges)
thick_hist, _ = np.histogram(thick_digit, bins=bin_edges)

# Normierung macht Histogramme unabhängig von der Bildgröße.
pixel_count = thin_digit.size
thin_features = thin_hist / pixel_count
thick_features = thick_hist / pixel_count

# Kurze Ausgabe zeigt die Merkmalsvektoren direkt.
print("Histogramm-Bins: dunkel, mittel, hell, sehr hell")
print("Dünner Strich:", np.round(thin_features, 3).tolist())
print("Dicker Strich:", np.round(thick_features, 3).tolist())

# Ein Balkendiagramm vergleicht die Histogrammmerkmale.
labels = ["dunkel", "mittel", "hell", "sehr hell"]
x_positions = np.arange(len(labels))
bar_width = 0.35

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x_positions - bar_width / 2, thin_features, bar_width, label="dünn")
ax.bar(x_positions + bar_width / 2, thick_features, bar_width, label="dick")

ax.set_title("Histogramme als kompakte Bildmerkmale")
ax.set_xlabel("Grauwertbereich")
ax.set_ylabel("Anteil der Pixel")
ax.set_xticks(x_positions)

ax.set_xticklabels(labels)
ax.set_ylim(0, 1)
ax.legend()
plt.show()



## **2. Kanten und HOG**

### **2.1. Kanten als Merkmale**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_A/image_02_01.jpg?v=1787650763" width="250">



>* Kanten zeigen starke Helligkeits- oder Farbwechsel
>* Sie beschreiben Formen robuster als Pixelwerte

>* Glättung reduziert Rauschen vor der Kantenerkennung
>* Kantenrichtungen liefern Merkmale für Klassifikatoren

>* Training und Test strikt getrennt halten
>* Kanten realistisch und robust bewerten



In [ ]:
#@title Python-Code - Kanten als Merkmale

# Wir nutzen Kanten als einfache Bildmerkmale.
# Ein Klassifikator lernt nur aus Trainingsmerkmalen.
# Die Testdaten bleiben bis zur Bewertung getrennt.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import sklearn

# Der kleine Zifferndatensatz ist offline verfügbar.
digits = load_digits()
images = digits.images.astype(np.float32)
labels = digits.target

# Diese Prüfung macht die Bildannahme sichtbar.
if images.shape[1:] != (8, 8):
    raise ValueError("Erwartet werden kleine Bilder mit 8 mal 8 Pixeln.")

# Sobel-Filter messen horizontale und vertikale Helligkeitsänderungen.
gradient_y, gradient_x = np.gradient(images, axis=(1, 2))
edge_strength = np.sqrt(gradient_x ** 2 + gradient_y ** 2)

# Jedes Kantenbild wird zu einem Merkmalsvektor abgeflacht.
edge_features = edge_strength.reshape(edge_strength.shape[0], -1)

# Die Aufteilung passiert vor Skalierung und Modelltraining.
X_train, X_test, y_train, y_test = train_test_split(
    edge_features, labels, test_size=0.25, stratify=labels, random_state=42
)

# Die Pipeline passt die Skalierung nur auf Trainingsdaten an.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)

# Das Modell sieht beim Lernen nur Trainingsmerkmale.
model.fit(X_train, y_train)
predicted = model.predict(X_test)

# Die Bewertung nutzt erst jetzt die zurückgehaltenen Testdaten.
accuracy = accuracy_score(y_test, predicted)
print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Kantenmerkmale pro Bild: {edge_features.shape[1]}")
print(f"Testgenauigkeit ohne Datenleckage: {accuracy:.3f}")

# Ein Testbild zeigt, welche Kantenmerkmale genutzt wurden.
example_index = 0
example_edges = edge_strength[example_index]

# Die einzelne Grafik zeigt das Kantenbild als Merkmalsquelle.
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(example_edges, cmap="gray")
ax.set_title("Kantenstärke eines Ziffernbildes")
ax.set_xlabel("Pixelspalte")
ax.set_ylabel("Pixelzeile")
plt.show()



### **2.2. HOG Merkmale**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_A/image_02_02.jpg?v=1787650761" width="250">



>* HOG beschreibt Bilder über lokale Kantenrichtungen
>* Kompakte Formmerkmale für klassische Klassifikatoren

>* Zellen sammeln gewichtete Kantenrichtungen.
>* Normalisierung macht Formen lichtrobuster.

>* HOG erzeugt feste Merkmalsvektoren pro Bild
>* Trainingssplit verhindert Datenleckage bei Vorverarbeitung



In [ ]:
#@title Python-Code - HOG Merkmale

# Dieses Beispiel trainiert ein kleines HOG-Bildmodell.
# HOG fasst lokale Kantenrichtungen als Merkmale zusammen.
# Die Auswertung zeigt Genauigkeit und typische Verwechslungen.

import numpy as np
import matplotlib.pyplot as plt
from sklearn import __version__ as sklearn_version
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import ConfusionMatrixDisplay

# Wir nutzen kleine Ziffernbilder aus scikit-learn.
digits = load_digits()
images = digits.images.astype(np.float32)
labels = digits.target

# Diese Prüfung macht die erwartete Bildform sichtbar.
if images.shape[1:] != (8, 8):
    raise ValueError("Erwartet werden kleine Bilder mit 8 mal 8 Pixeln.")

# HOG wird hier bewusst einfach und nachvollziehbar berechnet.
def simple_hog(image):
    gy, gx = np.gradient(image)
    magnitude = np.sqrt(gx * gx + gy * gy)
    angle = (np.degrees(np.arctan2(gy, gx)) + 180) % 180

    features = []
    for row in range(0, 8, 4):
        for col in range(0, 8, 4):
            cell_mag = magnitude[row:row + 4, col:col + 4]
            cell_ang = angle[row:row + 4, col:col + 4]
            hist, _ = np.histogram(
                cell_ang,
                bins=9,
                range=(0, 180),
                weights=cell_mag,
            )
            features.extend(hist)

    features = np.array(features, dtype=np.float32)
    norm = np.linalg.norm(features) + 1e-8
    return features / norm

# Jedes Bild wird vor dem Split in einen HOG-Vektor umgewandelt.
hog_features = np.array([simple_hog(image) for image in images])

# Der Split kommt vor der lernenden Standardisierung im Pipeline-Modell.
X_train, X_test, y_train, y_test = train_test_split(
    hog_features,
    labels,
    test_size=0.25,
    random_state=42,
    stratify=labels,
)

# Die Pipeline verhindert Datenleckage bei der Merkmalsskalierung.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=300, random_state=42),
)

# Nur Trainingsdaten passen Skalierung und Klassifikator an.
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"scikit-learn-Version: {sklearn_version}")
print(f"HOG-Merkmale pro Bild: {hog_features.shape[1]}")
print(f"Testgenauigkeit ohne Datenleckage: {accuracy:.3f}")

# Die Konfusionsmatrix zeigt, welche Ziffern verwechselt werden.
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    ax=ax,
    colorbar=False,
)
ax.set_title("Konfusionsmatrix für einfache HOG-Merkmale")
ax.set_xlabel("Vorhergesagte Ziffer")
ax.set_ylabel("Wahre Ziffer")
plt.tight_layout()
plt.show()



### **2.3. Leakage freier Split**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_A/image_02_03.jpg?v=1787650765" width="250">



>* Testdaten bleiben bis zum Schluss unbekannt
>* Vorverarbeitung nur mit Trainingsdaten anpassen

>* Lerne Vorverarbeitung nur auf Trainingsdaten
>* Trenne verwandte Bildquellen konsequent

>* Split vor allen Lernschritten festlegen
>* Testdaten nur zur ehrlichen Endbewertung



In [ ]:
#@title Python-Code - Leakage freier Split

# Wir demonstrieren einen leakagefreien Bilddaten-Split.
# Standardisierung wird nur am Training angepasst.
# Die Testbewertung bleibt dadurch ehrlicher.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay

# Wir laden kleine Ziffernbilder aus scikit-learn.
digits = load_digits()

# Jedes Bild wird zu einem einfachen Merkmalsvektor.
features = digits.images.reshape(len(digits.images), -1)
target = digits.target

# Eine kurze Prüfung macht die Annahmen sichtbar.
if features.shape[0] != target.shape[0]:
    raise ValueError("Bildanzahl und Zielwerte passen nicht zusammen.")

# Der Split passiert vor jeder lernenden Vorverarbeitung.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.25, stratify=target, random_state=42
)

# Die Pipeline passt den Skalierer nur auf Trainingsdaten an.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)

# Training nutzt ausschließlich Trainingsbilder und Trainingslabels.
model.fit(X_train, y_train)

# Die Testdaten werden erst nach dem Training bewertet.
test_accuracy = model.score(X_test, y_test)
y_pred = model.predict(X_test)

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Trainingsbilder: {len(X_train)}, Testbilder: {len(X_test)}")
print(f"Testgenauigkeit ohne Leakage: {test_accuracy:.3f}")

# Die Konfusionsmatrix zeigt typische Verwechslungen im Testset.
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, colorbar=False)
ax.set_title("Leakagefreier Split: Konfusionsmatrix")
ax.set_xlabel("Vorhergesagte Ziffer")
ax.set_ylabel("Wahre Ziffer")
plt.tight_layout()
plt.show()



## **3. Bildmodelle bewerten**

### **3.1. Klassifikatoren trainieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_A/image_03_01.jpg?v=1787650750" width="250">



>* Bildmerkmale helfen Klassen zuverlässig zu unterscheiden
>* Strikte Datentrennung verhindert Datenleckage

>* Klassifikatoren reagieren verschieden auf Merkmale
>* Testbilder erst nach sauberem Training bewerten

>* Konfusionsmatrix zeigt systematische Klassenverwechslungen
>* Fehlbilder erklären Ursachen und Verbesserungsbedarf



In [ ]:
#@title Python-Code - Klassifikatoren trainieren

# Wir trainieren einen einfachen Bildklassifikator.
# Saubere Pipelines verhindern Datenleckage beim Skalieren.
# Die Konfusionsmatrix zeigt typische Verwechslungen.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

# Wir laden kleine Ziffernbilder aus scikit-learn.
digits = load_digits()
images = digits.images
labels = digits.target

# Jedes Bild wird zu einem Merkmalsvektor abgeflacht.
features = images.reshape(images.shape[0], -1)

# Eine einfache Prüfung schützt vor unerwarteten Datenformen.
if features.shape[0] != labels.shape[0]:
    raise ValueError("Bildanzahl und Labelanzahl passen nicht zusammen.")

# Die Aufteilung bleibt stratifiziert und damit klassenfair.
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.25, stratify=labels, random_state=42
)

# Skalierung und Modelltraining passieren nur im Trainingsanteil.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)

# Die Pipeline verhindert Datenleckage automatisch.
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Wir berechnen Trefferquote und Konfusionsmatrix.
accuracy = accuracy_score(y_test, y_pred)
confusion = confusion_matrix(y_test, y_pred)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Testbilder: {len(y_test)}")
print(f"Trefferquote auf Testdaten: {accuracy:.3f}")

# Die Matrix zeigt, welche Ziffern verwechselt werden.
fig, ax = plt.subplots(figsize=(7, 6))
display = ConfusionMatrixDisplay(confusion_matrix=confusion)
display.plot(ax=ax, cmap="Blues", colorbar=False)

ax.set_title("Konfusionsmatrix für zurückgehaltene Ziffernbilder")
ax.set_xlabel("Vorhergesagte Klasse")
ax.set_ylabel("Wahre Klasse")
plt.tight_layout()
plt.show()



### **3.2. PCA Ergebnisse prüfen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_A/image_03_02.jpg?v=1787650752" width="250">



>* PCA verändert, welche Bildinformationen sichtbar bleiben
>* Kleine Datensätze können irreführende Muster erzeugen

>* PCA zeigt Klassentrennung und Überlappungen.
>* Konfusionsmatrix verbindet Kennzahlen mit Bildbeispielen.

>* PCA nur mit Trainingsdaten anpassen
>* Fehlerbilder zeigen Modellgrenzen und Datenmuster



In [ ]:
#@title Python-Code - PCA Ergebnisse prüfen

# Wir prüfen PCA-Ergebnisse an kleinen Ziffernbildern.
# Die Pipeline verhindert Datenleckage beim Trainieren.
# Fehlklassifikationen erscheinen sichtbar im PCA-Raum.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix

# Wir laden kleine Bilddaten aus scikit-learn.
digits = load_digits()
images = digits.images
labels = digits.target

# Jedes Bild wird zu einem Merkmalsvektor abgeflacht.
features = images.reshape(images.shape[0], -1)
if features.shape[0] != labels.shape[0]:
    raise ValueError("Bildanzahl und Labelanzahl passen nicht zusammen.")

# Der Split trennt Training und Test sauber.
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.25, stratify=labels, random_state=42
)

# Skalierung, PCA und Modell werden nur mit Training angepasst.
model = Pipeline(
    [("scale", StandardScaler()), ("pca", PCA(n_components=2)),
     ("clf", LogisticRegression(max_iter=1000, random_state=42))]
)

# Das Modell sieht beim Lernen keine Testbilder.
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Die PCA-Koordinaten der Testbilder entstehen unverändert.
test_points = model.named_steps["pca"].transform(
    model.named_steps["scale"].transform(X_test)
)

# Wir berechnen wenige Kennzahlen für die Bewertung.
accuracy = accuracy_score(y_test, y_pred)
confusion = confusion_matrix(y_test, y_pred)
errors = y_test != y_pred
error_count = int(np.sum(errors))

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Testgenauigkeit mit zwei PCA-Komponenten: {accuracy:.3f}")
print(f"Anzahl falsch klassifizierter Testbilder: {error_count}")
print(f"Häufigste Verwechslung: wahr {np.unravel_index(np.argmax(confusion - np.diag(np.diag(confusion))), confusion.shape)[0]}, vorhergesagt {np.unravel_index(np.argmax(confusion - np.diag(np.diag(confusion))), confusion.shape)[1]}")

# Die Grafik zeigt richtige und falsche Punkte im PCA-Raum.
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(test_points[~errors, 0], test_points[~errors, 1], s=18, alpha=0.55,
           label="richtig klassifiziert")
ax.scatter(test_points[errors, 0], test_points[errors, 1], s=45, marker="x",
           color="red", label="falsch klassifiziert")
ax.set_title("PCA-Raum: Wo liegen Fehlklassifikationen?")
ax.set_xlabel("erste Hauptkomponente")
ax.set_ylabel("zweite Hauptkomponente")
ax.legend()
plt.show()



### **3.3. Mini Projekt Bildbewertung**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_A/image_03_03.jpg?v=1787650754" width="250">



>* Modellentscheidungen über Genauigkeit hinaus verstehen
>* Konfusionsmatrix und Bilder gemeinsam auswerten

>* Fehlerbilder zeigen verborgene Muster
>* Merkmale erklären plausible Modellentscheidungen

>* Modellleistung im Anwendungskontext begründet einschätzen
>* Fehler analysieren und nächste Schritte ableiten



In [ ]:
#@title Python-Code - Mini Projekt Bildbewertung

# Wir bewerten ein klassisches Bildmodell.
# Die Konfusionsmatrix zeigt typische Verwechslungen.
# Fehlklassifikationen werden als Bilder sichtbar.

import numpy as np
import matplotlib.pyplot as plt
from sklearn import __version__ as sklearn_version

from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir laden kleine Ziffernbilder aus scikit-learn.
digits = load_digits()
images = digits.images
labels = digits.target

# Diese Prüfung macht die Bildform ausdrücklich sichtbar.
if images.shape[1:] != (8, 8):
    raise ValueError("Erwartet werden kleine 8-mal-8-Bilder.")

# Pixelbilder werden zu Merkmalsvektoren umgeformt.
features = images.reshape(images.shape[0], -1)

# Die Aufteilung trennt Training und Test sauber.
X_train, X_test, y_train, y_test, images_train, images_test = train_test_split(
    features, labels, images, test_size=0.25, stratify=labels, random_state=42
)

# Skalierung und Modell werden nur mit Trainingsdaten angepasst.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)

# Das Modell lernt aus den Trainingsmerkmalen.
model.fit(X_train, y_train)

# Danach bewerten wir nur auf ungesehenen Testdaten.
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

# Fehlklassifikationen werden für die Interpretation gezählt.
wrong_mask = y_pred != y_test
wrong_count = int(np.sum(wrong_mask))

# Ein typischer Fehler wird für die Bildansicht ausgewählt.
wrong_indices = np.flatnonzero(wrong_mask)
if len(wrong_indices) == 0:
    example_index = 0
else:
    example_index = int(wrong_indices[0])

# Kurze Ausgaben verbinden Kennzahl und Fehleranalyse.
print(f"scikit-learn-Version: {sklearn_version}")
print(f"Testgenauigkeit: {accuracy:.3f}")
print(f"Fehlklassifikationen im Testset: {wrong_count} von {len(y_test)}")
print(f"Beispielbild: wahr {y_test[example_index]}, vorhergesagt {y_pred[example_index]}")

# Eine Achse zeigt Matrix und Beispielbild gemeinsam.
fig, ax = plt.subplots(figsize=(7, 6))

# Die Konfusionsmatrix fasst alle Testentscheidungen zusammen.
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, ax=ax, cmap="Blues", colorbar=False
)

# Titel und Achsen erklären die Leserichtung.
ax.set_title("Konfusionsmatrix für Ziffernbilder")
ax.set_xlabel("Vorhergesagte Klasse")
ax.set_ylabel("Wahre Klasse")

# Das Beispielbild wird klein in die Matrix eingebettet.
small_image = images_test[example_index]
image_box = ax.inset_axes([0.62, 0.58, 0.28, 0.28])
image_box.imshow(small_image, cmap="gray_r")
image_box.set_title("Fehlerbeispiel", fontsize=9)
image_box.axis("off")

plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Klassische Bildmodelle**</font>


In this lecture, you learned to:
- Bereiten kleine Bilddaten als Merkmalsvektoren oder klassische Bildmerkmale vor. 
- Trainieren klassische Klassifikatoren auf Bildmerkmalen ohne Datenleckage. 
- Bewerten Bildmodelle mit Konfusionsmatrix und visualisierten Fehlklassifikationen. 

In the next Lecture (Lecture B), we will go over 'Signalmodelle'